# Задача: выявить корреляцию (или ее отсутствие) между размером словаря токенизатора и качеством модели (выраженном в f1) для задачи сентимент-анализа.
Предлагаемый план работ:
1. Ознакомиться с документацией и реализовать LightningDataModule (https://lightning.ai/docs/pytorch/stable/data/datamodule.html)
 - Загрузить датасет
 - Провести предобработку
 - Выбрать токенизатор (BPE, WordPiece, Unigram)
 - Реализовать collate_fn функцию токенизации для даталодера - encode
 - Добавить даталодеры
2. Ознакомиться с документацией и реализовать LightningModule (https://lightning.ai/docs/pytorch/LTS/common/lightning_module.html)
 - Выбрать и реализовать подходящую архитектуру модели для сентимент-анализа текста
 - Написать прямой проход модели
 - Написать training_step, validation_step, test_step
3. Провести цикл обучений модели
 - Ознакомиться с документацией и выбрать оптимальные настройки для Trainer (https://lightning.ai/docs/pytorch/stable//common/trainer.html)
 - Зафиксировать все гиперпараметры и провести обучение-валидацию-тестирование для 5 значений размера словаря (выбранных на ваше усмотрение)
 - Полученные результаты представить в виде гистограммы или любой другой визуализации

In [1]:
import torch
import datasets
import lightning.pytorch as pl
import tokenizers
from typing import *
import torchmetrics

ModuleNotFoundError: No module named 'lightning'

# Data

In [ ]:
class MyDataModule(pl.LightningDataModule):
    def __init__(
        self,
        ds_path: str,
        bs: int = 16,
        num_workers: int = 12,
        vocab_size: int = 5000,
        max_length: int = 256
    ):
        super().__init__()
        #TODO: download and process self.ds
        self.ds = datasets.DatasetDict(ds)

        #TODO: try Unigram instead of BPE
        tokenizer = tokenizers.Tokenizer(tokenizers.models.BPE(unk_token="[UNK]"))
        trainer = tokenizers.trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"])
        tokenizer.train_from_iterator(self.ds['train']['text'], trainer=trainer)

        tokenizer.enable_truncation(max_length)
        tokenizer.enable_padding(pad_token="[PAD]", direction="left")
        self.tokenizer = tokenizer

    def train_dataloader(self) -> torch.utils.data.DataLoader:
        #TODO: add DataLoader
        return None

    def val_dataloader(self) -> torch.utils.data.DataLoader:
        #TODO: add DataLoader
        return None

    def test_dataloader(self) -> torch.utils.data.DataLoader:
        #TODO: add DataLoader
        return None
    
    def encode(self, batch) -> Dict:
        #TODO: tokenize, pad and truncate
        return {"input_ids": input_ids, "label": label}

# Model

In [ ]:
class RNNModel(pl.LightningModule):
    def __init__(self, vocab_size, embd_size, hidden_size, output_size):
        super(RNNModel, self).__init__()
        #TODO: add your model
    
    def configure_model(self):
            self.metrics = {
                split: torchmetrics.F1Score(
                    task="multiclass", num_classes=self.output_size
                ) 
                for split in ["train", "val", "test"]
            }

    def forward(self, x):
        #TODO: add feedforward
        return out
    
    def training_step(self, batch, batch_idx):
        # TODO: report metrics and return loss
        return loss
    
    def validation_step(self, batch, batch_idx):
        # TODO: report metrics and return loss
        return loss
    
    def test_step(self, batch, batch_idx):
        # TODO: report metrics and return loss
        return loss
    
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=0.001)


# Trainer

In [ ]:
vocab_size = 1000
max_length = 256
embd_size = 256
hidden_size = 256

dm = MyDataModule(vocab_size=vocab_size, max_length=max_length)
model = RNNModel(vocab_size=vocab_size, embd_size=embd_size, hidden_size=hidden_size, output_size=len(dm.label_dict))

trainer = pl.Trainer(max_epochs=10)
trainer.fit(model=model, datamodule=dm)
trainer.test(datamodule=dm, ckpt_path="best")
#TODO: report plots here